In [1]:
!pip install transformers accelerate peft bitsandbytes datasets trl torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.1 MB/s  0:00:020m eta 0:00:01


1)transformers - to loads llama

2)accelerate - efficiently distributes training

3)peft - implements LoRA/QLoRA

4)bitsandbytes - enables 4-bit quantization

5)datasets - loads mlabonne/guanaco-llama2 dataset

6)trl - Supports fine-tuning for chat models

In [13]:
!pip install python-dotenv


In [1]:
from huggingface_hub import login
from dotenv import load_dotenv

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "meta-llama/Llama-3.2-1B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto"
)



W1007 21:39:38.592000 3714 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1007 21:39:38.621000 3714 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [3]:
#Loading the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
print(tokenizer)

TokenizersBackend(name_or_path='meta-llama/Llama-3.2-1B', vocab_size=128000, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>'}, added_tokens_decoder={
	128000: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128001: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128002: AddedToken("<|reserved_special_token_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128003: AddedToken("<|reserved_special_token_1|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128004: AddedToken("<|finetune_right_pad_id|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128005: AddedToken("<|reserved_special_token_2|>", rstrip=False, lstrip=False, single_word=False, normalized=Fa

In [4]:
# Setting the padding token to be the same as the end-of-sequence (EOS) token
# This is often necessary because some models don't have a predefined padding token
tokenizer.pad_token = tokenizer.eos_token

In [5]:
from datasets import load_dataset

dataset = load_dataset("mlabonne/guanaco-llama2", split="train")

print(dataset[0])

{'text': '<s>[INST] Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research. [/INST] "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.\n\nRecent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, workers often face low wages, limited benefits, and reduced bargaining power, le

In [6]:
{
  "text": "### Instruction: Explain fine-tuning in simple terms. \n### Response: Fine-tuning means adapting a pre-trained model to perform better on a specific task."
}

{'text': '### Instruction: Explain fine-tuning in simple terms. \n### Response: Fine-tuning means adapting a pre-trained model to perform better on a specific task.'}

In [7]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length = 512)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/9846 [00:00<?, ? examples/s]

In [8]:
#Applying LoRA adapters
from peft import  LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [9]:
#Apply LoRA to the pretrained model
model = get_peft_model(model, lora_config)


In [10]:
#Printing the no of trainable parameters in the model
model.print_trainable_parameters()

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [11]:
from transformers import TrainingArguments
#Defining the training configuration using TrainingArguments
training_args = TrainingArguments(
    output_dir="'./llama-finetuned", #Dir where trained model and checkpoints will be saved
    gradient_checkpointing=True,
    per_device_train_batch_size=1, #Batch size per device (eg;GPU).
    gradient_accumulation_steps=4, #No of steps to accumalate gradients before updating weights.#This simulated a larger batch size without increasing memory usage.
    optim="adamw_bnb_8bit", # Optimizer to use.Here, it's AdamW with 8-bit quantization for memory efficiency
    save_total_limit=2, #Max no of model checkpoints to save.Older checkpoints are deleted.
    save_strategy="epoch", #Safe a checkpoint at the end of each epoch
    learning_rate=2e-4, # Learning rate for the optimizer. Controls how quickly the model learns
    num_train_epochs=3, # No of epochs to train the model. One epoch = one full pass through the dataset.
    logging_dir="./logs", #Dir where training logs will be stored
    fp16=True, # Enable mixed-precision training to reduce memory usage and speed up training
    push_to_hub=True, # Whether to push the trained model to the Hugging Face Hub. Set to False for local training
    report_to="none" #Disable reporting to external services like weights & Biases or TensorBoard
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
# Import the necessary classes from the hugging Face Transformers library

from transformers import Trainer, DataCollatorForLanguageModeling

#Creating a data collector for language modeling tasks
#This prepares batches of data for training by handling padding and other preprocessing steps
data_collactor = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, #The tokenizer used to preprocess the data
    mlm = False # Whether to use masked language modeling (MLM). Set to False for causal langauge models
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collactor
)

In [13]:
trainer.train()

Step,Training Loss
500,1.707037
1000,1.686017
1500,1.659352
2000,1.671788
2500,1.634546
3000,1.631605
3500,1.623583
4000,1.620646
4500,1.596611
5000,1.617482


/home/ash/pygame_310/lib/python3.10/site-packages/peft/utils/other.py:1508: UserWarning: Unable to fetch remote file due to the following error [Errno 104] Connection reset by peer - silently ignoring the lookup for the file config.json in meta-llama/Llama-3.2-1B.
  warnings.warn(
/home/ash/pygame_310/lib/python3.10/site-packages/peft/utils/save_and_load.py:313: UserWarning: Could not find a config file in meta-llama/Llama-3.2-1B - will assume that the vocabulary was not modified.
  warnings.warn(
'[Errno 104] Connection reset by peer' thrown while requesting POST https://huggingface.co/api/models/ashkck/llama-finetuned/preupload/main
Retrying in 1s [Retry 1/5].


TrainOutput(global_step=7386, training_loss=1.625773715224009, metrics={'train_runtime': 9646.4561, 'train_samples_per_second': 3.062, 'train_steps_per_second': 0.766, 'total_flos': 8.845861084515533e+16, 'train_loss': 1.625773715224009, 'epoch': 3.0})

In [14]:
model.save_pretrained("./llama3.2-finetuned")
tokenizer.save_pretrained("./llama3.2-finetuned")

('./llama3.2-finetuned/tokenizer_config.json',
 './llama3.2-finetuned/tokenizer.json')

In [15]:
model_path = "/home/ash/Projects/Fine-tuning Llama-3.2-1B/llama3.2-finetuned"

model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/64 [00:00<?, ?it/s]

In [16]:
test_dataset = load_dataset("mlabonne/guanaco-llama2", split="test")

In [17]:
len(test_dataset)

518

In [18]:
test_dataset[0]

{'text': '<s>[INST] Напиши функцию на языке swift, которая сортирует массив целых чисел, а затем выводит его на экран [/INST] Вот функция, которая сортирует массив целых чисел и выводит его на экран:\n\n```swift\nfunc sortAndPrintArray(_ array: [Int]) {\n  // Создаем копию массива, чтобы не изменять исходный\n  var sortedArray = array\n  // Сортируем массив по возрастанию\n  sortedArray.sort()\n  // Выводим отсортированный массив на экран\n  print(sortedArray)\n}\n```\n\n\nТы можешь проверить работу функции, вызвав ее с любым массивом целых чисел, например:\n\n```swift\nsortAndPrintArray([5, 2, 7, 9, -3])\n```\n\nНа экране появится:\n\n```sh\n[-3, 2, 5, 7, 9]\n```\n---\nНадеюсь, что я смог тебе помочь. Нужно ли тебе что-нибудь ещё? 😊 </s>'}